In [ ]:
%cd ../
%load_ext autoreload
%autoreload 2

# Brandenburg Renewable Energy Assets Mapping

This notebook plots the geographic locations of renewable energy assets across the State of Brandenburg, Germany:
1. **Wind Turbines**
2. **Ground-Stationed Solar Panels** (Freiflächensolaranlagen)
3. **Private-Household / Residential Solar Panels** (Gebäudesolaranlagen / other rooftop/balcony units)

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import HeatMap, MarkerCluster
import matplotlib.pyplot as plt

# 1. Load the administrative boundary of Brandenburg
boundary_path = "./data/external/brandenburg_boundary.geojson"
boundary = gpd.read_file(boundary_path)

# 2. Load wind turbines raw dataset
wind_df = pd.read_csv("./data/raw/mastr_wind_brandenburg_raw.csv")
wind_gdf = gpd.GeoDataFrame(
    wind_df, 
    geometry=gpd.points_from_xy(wind_df['longitude'], wind_df['latitude']),
    crs="EPSG:4326"
)

# 3. Load solar installations raw dataset
solar_df = pd.read_csv("./data/raw/mastr_solar_brandenburg_raw.csv")
solar_gdf = gpd.GeoDataFrame(
    solar_df, 
    geometry=gpd.points_from_xy(solar_df['longitude'], solar_df['latitude']),
    crs="EPSG:4326"
)

# Split solar into ground-mounted and residential
ground_solar = solar_gdf[solar_gdf['is_ground_mounted'] == True]
res_solar = solar_gdf[solar_gdf['is_ground_mounted'] == False]

print(f"Loaded {len(wind_gdf)} wind turbines.")
print(f"Loaded {len(ground_solar)} ground-mounted solar installations.")
print(f"Loaded {len(res_solar)} residential solar installations.")

## Map 1: Wind Turbines in Brandenburg

In [ ]:
# Static visualization showing the spatial distribution
fig, ax = plt.subplots(figsize=(10, 10))
boundary.plot(ax=ax, color='whitesmoke', edgecolor='black', linewidth=1.5)
wind_gdf.plot(ax=ax, color='teal', markersize=4, alpha=0.6, label='Wind Turbines')
ax.set_title('Wind Turbines in Brandenburg', fontsize=14)
ax.axis('off')
plt.legend()
plt.show()

# Interactive folium map with Marker Clustering
m_wind = folium.Map(location=[52.35, 13.0], zoom_start=8, tiles='OpenStreetMap')
folium.GeoJson(
    boundary, 
    style_function=lambda x: {'fillColor': '#ffffff', 'color': '#000000', 'weight': 2, 'fillOpacity': 0.05}
).add_to(m_wind)

marker_cluster = MarkerCluster(name="Wind Turbines").add_to(m_wind)
for idx, row in wind_df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color='teal',
        fill=True,
        fill_color='teal',
        fill_opacity=0.6,
        popup=f"Unit: {row['unit_mastr_number']}<br>Power: {row['gross_power']} kW<br>Commissioned: {row['commissioning_date']}"
    ).add_to(marker_cluster)

m_wind

## Map 2: Ground-Mounted Solar Installations in Brandenburg

In [ ]:
# Static visualization
fig, ax = plt.subplots(figsize=(10, 10))
boundary.plot(ax=ax, color='whitesmoke', edgecolor='black', linewidth=1.5)
ground_solar.plot(ax=ax, color='darkorange', markersize=5, alpha=0.7, label='Ground Solar')
ax.set_title('Ground-Mounted Solar Installations in Brandenburg', fontsize=14)
ax.axis('off')
plt.legend()
plt.show()

# Interactive folium map with individual Markers
m_solar_ground = folium.Map(location=[52.35, 13.0], zoom_start=8, tiles='OpenStreetMap')
folium.GeoJson(
    boundary, 
    style_function=lambda x: {'fillColor': '#ffffff', 'color': '#000000', 'weight': 2, 'fillOpacity': 0.05}
).add_to(m_solar_ground)

for idx, row in ground_solar.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color='darkorange',
        fill=True,
        fill_color='darkorange',
        fill_opacity=0.7,
        popup=f"Unit: {row['unit_mastr_number']}<br>Power: {row['gross_power']} kW<br>Commissioned: {row['commissioning_date']}"
    ).add_to(m_solar_ground)

m_solar_ground

## Map 3: Private-Household / Residential Solar Panels in Brandenburg

In [ ]:
# Static visualization (handles 172k points extremely fast)
fig, ax = plt.subplots(figsize=(10, 10))
boundary.plot(ax=ax, color='whitesmoke', edgecolor='black', linewidth=1.5)
res_solar.plot(ax=ax, color='gold', markersize=0.2, alpha=0.1, label='Residential Solar')
ax.set_title('Private Household / Residential Solar Panels in Brandenburg', fontsize=14)
ax.axis('off')
plt.legend()
plt.show()

# Interactive Density Heatmap (handles 172k points efficiently without crashing the browser)
m_solar_res = folium.Map(location=[52.35, 13.0], zoom_start=8, tiles='OpenStreetMap')
folium.GeoJson(
    boundary, 
    style_function=lambda x: {'fillColor': '#ffffff', 'color': '#000000', 'weight': 2, 'fillOpacity': 0.05}
).add_to(m_solar_res)

heat_data = res_solar[['latitude', 'longitude']].values.tolist()
HeatMap(heat_data, radius=12, blur=8, max_zoom=10).add_to(m_solar_res)

m_solar_res